# Task 2.2 통합 분석: WHAT × WHEN × WHO

> 데이터: `si_dataset/review_for_analysis.json` + 팀원 클러스터 결과 (`주빈task2.2/`)

## 분석 스토리

```
[FOUNDATION] 별점을 믿지 마라 — 감성 재라벨링으로 숨겨진 불만 발굴
       ↓
[WHAT]  불만의 구조 — 6개 유형 클러스터 + 브랜드별 분포 + 연쇄 체인
       ↓
[WHEN]  불만의 타이밍 — 구매 코호트 × 월별 시계열 (배송 vs 카메라 패턴)
       ↓
[WHO]   불만의 주체 — 이메일 도메인 = 디지털 라이프스타일 프록시
       ↓
[BRIDGE] WHAT × WHEN, WHAT × WHO 교차 분석 (3개 분석 연결)
       ↓
[SYNTHESIS] 비즈니스 액션 3가지
```

## 슬라이드 매핑 (7분 발표용)

| 셀 섹션 | 발표 슬라이드 | 소요 시간 |
|---------|------------|----------|
| FOUNDATION | Slide 1 — Hook | 0:00–0:40 |
| WHAT | Slide 2 — 불만 구조 | 0:40–2:10 |
| WHEN | Slide 3 — 타이밍 | 2:10–3:40 |
| WHO | Slide 4 — 소비자 세그먼트 | 3:40–5:10 |
| BRIDGE | Slide 5 — 교차 분석 | 5:10–6:20 |
| SYNTHESIS | Slide 6 — 비즈니스 액션 | 6:20–7:00 |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 0. SETUP
# ══════════════════════════════════════════════════════════════════
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter
from itertools import combinations
from scipy import stats

warnings.filterwarnings('ignore')

# ── 재현성 시드 고정 ──────────────────────────────────────────────
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── 한글 폰트 ─────────────────────────────────────────────────────
_font_path = r'C:\Windows\Fonts\malgun.ttf'
fm.fontManager.addfont(_font_path)
_font_name = fm.FontProperties(fname=_font_path).get_name()
plt.rcParams['font.family'] = _font_name
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted', rc={
    'font.family': _font_name, 'axes.unicode_minus': False
})

# ── 경로 설정 ─────────────────────────────────────────────────────
BASE_DIR   = Path('주빈task2.2')
DATA_PATH  = Path('si_dataset/review_for_analysis.json')
OUT_DIR    = Path('output/integrated')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 클러스터 상수 ──────────────────────────────────────────────────
CLUSTER_NAMES = {
    0: '혼재',
    1: '쿠팡_배송포장',
    2: '쿠팡_교환절차',
    3: '제품_완성도+배터리',
    4: '제품_카메라+화면',
    5: '가격_구성품',
}
CLUSTER_COLORS = {
    '혼재':              '#95a5a6',
    '쿠팡_배송포장':      '#e74c3c',
    '쿠팡_교환절차':      '#e67e22',
    '제품_완성도+배터리': '#3498db',
    '제품_카메라+화면':   '#9b59b6',
    '가격_구성품':        '#27ae60',
}
MAIN_CLUSTERS = ['쿠팡_배송포장', '쿠팡_교환절차', '제품_완성도+배터리', '제품_카메라+화면', '가격_구성품']

COHORT_ORDER  = ['D+0~30 (얼리어답터)', 'D+31~90 (초기다수)', 'D+91+ (후기다수)']
COHORT_COLORS = {
    'D+0~30 (얼리어답터)': '#2196F3',
    'D+31~90 (초기다수)':  '#FF9800',
    'D+91+ (후기다수)':    '#9C27B0',
}

PRODUCT_LABELS = {
    'iphone_17': 'iPhone 17',
    'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26',
    'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7',
    'galaxy_z_flip7': 'Galaxy Z Flip7',
}

print('Setup complete. 출력 디렉토리:', OUT_DIR.resolve())

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 1. DATA LOADING
# ══════════════════════════════════════════════════════════════════

# ── 1-a. 원본 리뷰 데이터 (메타데이터 + 날짜 + member 포함) ──────
# review_with_sentiment.csv = 원본 JSON을 팀원이 전처리한 버전
# (reviewAt, member, product_name, rating, helpfulCount 등 모든 컬럼 포함)
df = pd.read_csv(BASE_DIR / 'review_with_sentiment.csv')
df['reviewAt_dt'] = pd.to_datetime(df['reviewAt'], unit='ms')
df['review_month'] = df['reviewAt_dt'].dt.to_period('M')
df['brand'] = df['product_name'].apply(lambda x: 'Apple' if 'iphone' in str(x) else 'Samsung')
df['product_label'] = df['product_name'].map(PRODUCT_LABELS)
df['content_len'] = df['content'].fillna('').str.len()

# reviewId 타입 통일 (int)
df['reviewId'] = df['reviewId'].astype(int)

print(f'전체 리뷰: {len(df):,}개')
print(f'기간: {df["reviewAt_dt"].min().date()} ~ {df["reviewAt_dt"].max().date()}')
print(f'제품: {df["product_name"].unique().tolist()}')

# ── 1-b. 불만 클러스터 결과 (팀원 분석) ──────────────────────────
cc = pd.read_csv(BASE_DIR / 'clustered_complaints.csv')
cc['cluster_name'] = cc['cluster'].map(CLUSTER_NAMES)
cc['review_id'] = cc['review_id'].astype(int)

print(f'\n불만 문장 클러스터: {len(cc):,}개')
print(cc['cluster_name'].value_counts().to_string())

# ── 1-c. 문장 수준 감성 결과 (전체 문장, 5점 숨겨진 불만 분석용) ──
ss = pd.read_csv(BASE_DIR / 'sentence_sentiments.csv')
print(f'\n전체 문장: {len(ss):,}개  |  부정 문장: {ss["is_negative"].sum():,}개 ({ss["is_negative"].mean()*100:.1f}%)')

## 감성 분석 결과 플러그인

아래 `SENTIMENT_FILE` 에 파일 경로를 지정하면 사용자 제공 감성 결과를 사용합니다.

**필요 컬럼:**
- `review_id` (int) — `reviewId`와 동일한 값
- `pred_label_string` — `'강한긍정'` / `'약한긍정'` / `'약한부정'` / `'강한부정'`

파일이 없으면 팀원 기존 결과(`review_with_sentiment.csv`)로 자동 대체됩니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# [SENTIMENT PLUG-IN] 아래 경로를 수정하세요
# ══════════════════════════════════════════════════════════════════
SENTIMENT_FILE = None   # 예: 'my_sentiment_results.csv'
# ══════════════════════════════════════════════════════════════════

KO_LABEL_MAP = {
    '강한긍정':    'positive',
    '약한긍정':    'weak_positive',
    '약한부정':    'weak_negative',
    '강한부정':    'negative',
    'positive':    'positive',
    'weak_positive': 'weak_positive',
    'weak_negative': 'weak_negative',
    'negative':    'negative',
}

if SENTIMENT_FILE and Path(SENTIMENT_FILE).exists():
    user_sent = pd.read_csv(SENTIMENT_FILE)
    user_sent = user_sent.rename(columns={'review_id': 'reviewId'})
    user_sent['reviewId'] = user_sent['reviewId'].astype(int)
    user_sent['pred_label_string'] = user_sent['pred_label_string'].map(KO_LABEL_MAP)
    df = df.drop(columns=['pred_label_string'], errors='ignore')
    df = df.merge(user_sent[['reviewId', 'pred_label_string']], on='reviewId', how='left')
    print('[사용자 감성 결과 로드 완료]')
else:
    # fallback: 팀원 기존 결과 사용 (pred_label_string 이미 df에 있음)
    df['pred_label_string'] = df['pred_label_string'].map(KO_LABEL_MAP)
    print('[기존 팀원 감성 결과 사용 (fallback)]')

df['pred_label_string'] = df['pred_label_string'].fillna('weak_positive')
df['is_negative'] = df['pred_label_string'].isin({'negative', 'weak_negative'})

# 한글 레이블 파생
LABEL_EN2KO = {
    'negative':    '강한부정',
    'weak_negative': '약한부정',
    'weak_positive': '약한긍정',
    'positive':    '강한긍정',
}
df['sentiment_ko'] = df['pred_label_string'].map(LABEL_EN2KO)

print(f'\n감성 분포:')
print(df['sentiment_ko'].value_counts().to_string())
print(f'\n부정 리뷰: {df["is_negative"].sum():,}개 ({df["is_negative"].mean()*100:.1f}%)')

---
## [FOUNDATION] 별점을 믿지 마세요 — Slide 1

In [ ]:
LABEL_KO_ORDER = ['강한부정', '약한부정', '약한긍정', '강한긍정']

ct = pd.crosstab(df['rating'], df['sentiment_ko'])
ct = ct.reindex(columns=[c for c in LABEL_KO_ORDER if c in ct.columns])
ct_norm = ct.div(ct.sum(axis=1), axis=0) * 100

# 5점 리뷰 안 숨겨진 부정 문장 수
hidden_complaints = ss[(ss['rating'] == 5) & (ss['is_negative'] == True)]
n_hidden = len(hidden_complaints)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# ── 왼쪽: 원시 카운트 히트맵 ──────────────────────────────────────
sns.heatmap(ct, annot=True, fmt='d', cmap='Reds',
            linewidths=0.5, ax=axes[0],
            cbar_kws={'label': '리뷰 수'})
axes[0].set_title('Rating × 감성 예측 (원시 카운트)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('감성 예측 레이블')
axes[0].set_ylabel('별점 (Rating)')

# ── 오른쪽: 행 기준 비율 히트맵 ──────────────────────────────────
ax = axes[1]
sns.heatmap(ct_norm, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=0, vmax=100, linewidths=0.5, ax=ax,
            cbar_kws={'label': '비율 (%)'})
ax.set_title('Rating × 감성 예측 (행 기준 %)', fontsize=12, fontweight='bold')
ax.set_xlabel('감성 예측 레이블')
ax.set_ylabel('')

# 핵심 수치 강조 (5점 리뷰 오염)
rate5_neg_pct = ct_norm.loc[5, '강한부정':'약한부정'].sum() if 5 in ct_norm.index else 0
ax.text(0.5, 1.07,
        f'★★★★★(5점) 리뷰에서 부정 문장 {n_hidden:,}개 → 별점은 "대부분 만족"처럼 보이지만',
        transform=ax.transAxes, ha='center', fontsize=10,
        color='#c0392b', fontweight='bold')

plt.suptitle('[FOUNDATION] "별점을 믿지 마세요"\n'
             'Task 2.1 앙상블 모델 → 텍스트 기반 감성 재라벨링 → Rating 왜곡 제거',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide1_foundation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'=== Foundation 핵심 수치 ===')
print(f'리뷰 수준 부정 탐지: {df["is_negative"].sum():,}개')
print(f'문장 수준으로 내려가면: {ss["is_negative"].sum():,}개 ({ss["is_negative"].sum()/len(ss)*100:.1f}%)')
print(f'5점 리뷰 안 숨겨진 부정 문장: {n_hidden:,}개')
print(f'Rating 기반 탐지(1-2점): {(df["rating"]<=2).sum():,}개 vs 모델 탐지: {df["is_negative"].sum():,}개')

---
## [WHAT] 불만의 구조 — Slide 2

팀원 클러스터 분석 결과를 시각화합니다.

In [ ]:
# ── WHAT-1: UMAP 2D + 클러스터 규모 ──────────────────────────────
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from umap import UMAP

    print('TF-IDF 임베딩 + UMAP 계산 중... (1~2분 소요)')
    tfidf = TfidfVectorizer(
        token_pattern=r'[가-힣]{2,}',
        max_features=2000,
        sublinear_tf=True,
    )
    X = tfidf.fit_transform(cc['sentence'].fillna(''))

    umap_2d = UMAP(
        n_components=2, n_neighbors=15, min_dist=0.05,
        metric='cosine', random_state=SEED
    )
    coords = umap_2d.fit_transform(X)
    cc['umap_x'] = coords[:, 0]
    cc['umap_y'] = coords[:, 1]
    has_umap = True
    print('UMAP 완료.')

except ImportError:
    print('umap-learn 없음 → UMAP 시각화 생략, bar chart로 대체')
    has_umap = False

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── 왼쪽: UMAP 또는 대체 시각화 ──────────────────────────────────
ax1 = axes[0]
if has_umap:
    for cnum, cname in CLUSTER_NAMES.items():
        mask = cc['cluster'] == cnum
        ax1.scatter(
            cc.loc[mask, 'umap_x'], cc.loc[mask, 'umap_y'],
            c=CLUSTER_COLORS[cname],
            label=f'{cname} ({mask.sum():,}개)',
            alpha=0.35, s=7, edgecolors='none',
        )
        if mask.sum() > 10:
            cx = cc.loc[mask, 'umap_x'].mean()
            cy = cc.loc[mask, 'umap_y'].mean()
            ax1.annotate(
                cname, (cx, cy), fontsize=8, fontweight='bold', ha='center',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.85, edgecolor='gray'),
            )
    ax1.set_title('불만 문장 UMAP 2D\n(TF-IDF + UMAP 차원 축소)', fontsize=12, fontweight='bold')
    ax1.legend(markerscale=3, fontsize=8, loc='best')
    ax1.set_xlabel('UMAP 1')
    ax1.set_ylabel('UMAP 2')
else:
    # UMAP 없을 때 클러스터별 평균 별점 산점도
    cluster_stats = cc.groupby('cluster_name').agg(
        count=('cluster', 'size'),
        avg_rating=('rating', 'mean')
    ).reset_index()
    for _, row in cluster_stats.iterrows():
        ax1.scatter(row['avg_rating'], row['count'],
                    s=row['count']/3, c=CLUSTER_COLORS[row['cluster_name']],
                    alpha=0.8, edgecolors='white', linewidth=1.5)
        ax1.annotate(row['cluster_name'], (row['avg_rating'], row['count']),
                     fontsize=8, ha='center', va='bottom')
    ax1.set_title('클러스터별 규모 × 평균 별점', fontsize=12, fontweight='bold')
    ax1.set_xlabel('평균 별점')
    ax1.set_ylabel('문장 수')

# ── 오른쪽: 클러스터 규모 bar ─────────────────────────────────────
ax2 = axes[1]
cluster_sizes = (
    cc.groupby('cluster_name')
    .agg(count=('cluster', 'size'), avg_rating=('rating', 'mean'))
    .sort_values('count')
)
bar_colors = [CLUSTER_COLORS[n] for n in cluster_sizes.index]
bars = ax2.barh(
    cluster_sizes.index, cluster_sizes['count'],
    color=bar_colors, edgecolor='white', linewidth=1.5
)
for bar, (cname, row) in zip(bars, cluster_sizes.iterrows()):
    ax2.text(
        bar.get_width() + 10,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['count']):,}개  ({row['count']/len(cc)*100:.1f}%)  ★{row['avg_rating']:.2f}",
        va='center', fontsize=9,
    )
ax2.set_title('불만 클러스터 규모 + 평균 별점', fontsize=12, fontweight='bold')
ax2.set_xlabel('문장 수')
ax2.set_xlim(0, cluster_sizes['count'].max() * 1.5)

plt.suptitle('[WHAT] 불만의 구조 — 6개 유형으로 자동 분류', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide2a_what_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── WHAT-2: 브랜드 / 제품별 불만 클러스터 분포 ────────────────────
cc_merged = cc.merge(
    df[['reviewId', 'brand', 'product_label', 'rating']].rename(columns={'rating': 'rating_review'}),
    left_on='review_id', right_on='reviewId', how='left',
)

# 혼재(C0) 제외 — 신뢰도 낮음 (kss 분리 오류 다수)
cc_main = cc_merged[cc_merged['cluster_name'].isin(MAIN_CLUSTERS)].copy()

brand_cluster = (
    pd.crosstab(cc_main['brand'], cc_main['cluster_name'], normalize='index') * 100
).reindex(columns=[c for c in MAIN_CLUSTERS if c in cc_main['cluster_name'].unique()])

prod_cluster = (
    pd.crosstab(cc_main['product_label'], cc_main['cluster_name'], normalize='index') * 100
).reindex(columns=[c for c in MAIN_CLUSTERS if c in cc_main['cluster_name'].unique()])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(brand_cluster, annot=True, fmt='.1f', cmap='YlOrRd',
            vmin=0, vmax=45, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': '비율 (%)'})
axes[0].set_title('브랜드 × 불만 유형\n(행 기준 %)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('브랜드')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(prod_cluster, annot=True, fmt='.1f', cmap='YlOrRd',
            vmin=0, vmax=45, linewidths=0.5, ax=axes[1],
            cbar_kws={'label': '비율 (%)'})
axes[1].set_title('제품 × 불만 유형\n(행 기준 %)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('제품')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('[WHAT] 핵심 발견: 불만의 34%는 쿠팡 문제 / S26 Ultra 카메라 역설',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide2b_what_brand_product.png', dpi=150, bbox_inches='tight')
plt.show()

# 핵심 수치 출력
coupang_pct = len(cc[cc['cluster'].isin([1, 2])]) / len(cc) * 100
print(f'쿠팡 관련 클러스터(배송+교환) 비율: {coupang_pct:.1f}%')
print(f'\n[Apple 상위 불만]')
print(brand_cluster.loc['Apple'].sort_values(ascending=False).head(3).to_string())
print(f'\n[Samsung 상위 불만]')
print(brand_cluster.loc['Samsung'].sort_values(ascending=False).head(3).to_string())

In [ ]:
# ── WHAT-3: 연쇄 불만 체인 분석 (공존 패턴) ──────────────────────
review_clusters = (
    cc_main.groupby('reviewId')['cluster_name']
    .apply(lambda x: frozenset(x))
    .reset_index()
)
review_clusters.columns = ['reviewId', 'complaint_types']
multi = review_clusters[review_clusters['complaint_types'].apply(len) >= 2]

cooc = Counter()
for types in multi['complaint_types']:
    for pair in combinations(sorted(types), 2):
        cooc[pair] += 1

# 공존 히트맵
cooc_matrix = pd.DataFrame(0, index=MAIN_CLUSTERS, columns=MAIN_CLUSTERS)
for (a, b), cnt in cooc.items():
    if a in MAIN_CLUSTERS and b in MAIN_CLUSTERS:
        cooc_matrix.loc[a, b] = cnt
        cooc_matrix.loc[b, a] = cnt

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# 히트맵
mask = np.eye(len(MAIN_CLUSTERS), dtype=bool)
sns.heatmap(cooc_matrix, annot=True, fmt='d', cmap='Blues',
            mask=mask, ax=axes[0], linewidths=0.5,
            cbar_kws={'label': '공존 리뷰 수'})
axes[0].set_title('불만 유형 공존 히트맵\n(같은 리뷰에서 함께 등장)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)
axes[0].tick_params(axis='y', rotation=0)

# Top 조합 bar chart
top_pairs = pd.DataFrame(
    [(f"{a}\n+ {b}", cnt) for (a, b), cnt in cooc.most_common(6) if a in MAIN_CLUSTERS and b in MAIN_CLUSTERS],
    columns=['조합', '건수']
)
ax2 = axes[1]
bars = ax2.barh(
    top_pairs['조합'][::-1], top_pairs['건수'][::-1],
    color='#3498db', edgecolor='white', linewidth=1.2
)
for bar, v in zip(bars, top_pairs['건수'][::-1].values):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
             f'{v}건', va='center', fontsize=10)
ax2.set_title('연쇄 불만 조합 Top 6\n"카메라 불량→교환→CS" 체인 확인',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('공존 리뷰 수')
ax2.set_xlim(0, top_pairs['건수'].max() * 1.3)

plt.suptitle('[WHAT] 연쇄 불만 체인 — 하나의 리뷰에서 여러 불만이 동시에 폭발',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide2c_what_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'2개 이상 불만 유형 공존 리뷰: {len(multi):,}건')
print(f'\nTop 공존 조합:')
for (a, b), cnt in cooc.most_common(5):
    if a in MAIN_CLUSTERS and b in MAIN_CLUSTERS:
        print(f'  {a} + {b}: {cnt}건')

---
## [WHEN] 불만의 타이밍 — Slide 3

구매 코호트 정의 → 월별 시계열 → 배송 vs 카메라 비교

In [ ]:
# ── WHEN-1: 코호트 정의 ──────────────────────────────────────────
# 제품별 '첫 리뷰 날짜'를 D+0 기준점으로 사용 (공식 출시일 미활용)
first_review_dt = df.groupby('product_name')['reviewAt_dt'].min().to_dict()
df['release_date'] = df['product_name'].map(first_review_dt)
df['days_since_release'] = (df['reviewAt_dt'] - df['release_date']).dt.days

def assign_cohort(d):
    if pd.isna(d) or d < 0:
        return '출시 전'
    if d <= 30:
        return 'D+0~30 (얼리어답터)'
    if d <= 90:
        return 'D+31~90 (초기다수)'
    return 'D+91+ (후기다수)'

df['cohort'] = df['days_since_release'].apply(assign_cohort)
df_cohort = df[df['cohort'] != '출시 전'].copy()

print('제품별 D+0 기준점 (첫 리뷰 날짜):')
for prod, dt in sorted(first_review_dt.items()):
    print(f'  {PRODUCT_LABELS.get(prod, prod):<22}: {dt.strftime("%Y-%m-%d")}')

print(f'\n코호트 분포:')
print(df_cohort['cohort'].value_counts().to_string())

print(f'\n코호트별 평균 평점:')
for cohort in COHORT_ORDER:
    sub = df_cohort[df_cohort['cohort'] == cohort]
    print(f'  {cohort}: 평균 {sub["rating"].mean():.2f}  '
          f'부정률 {sub["is_negative"].mean()*100:.1f}%  (n={len(sub)})')

In [ ]:
# ── WHEN-2: 월별 불만 트렌드 + 배송 vs 카메라 대비 ────────────────

# 클러스터 할당에 날짜 정보 병합
cc_when = cc_main.merge(
    df[['reviewId', 'reviewAt_dt', 'cohort', 'days_since_release']],
    on='reviewId', how='left'
)
cc_when['review_month_str'] = cc_when['reviewAt_dt'].dt.to_period('M').astype(str)
cc_when = cc_when[cc_when['review_month_str'].notna()].copy()

# 월별 클러스터별 집계
monthly = (
    cc_when.groupby(['review_month_str', 'cluster_name'])
    .size().unstack(fill_value=0)
    .sort_index()
)
monthly = monthly.reindex(columns=[c for c in MAIN_CLUSTERS if c in monthly.columns])

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# ── 상단: 전체 월별 stacked bar ────────────────────────────────────
ax1 = axes[0]
x = np.arange(len(monthly))
month_labels = list(monthly.index)
bottom = np.zeros(len(monthly))

for cname in monthly.columns:
    ax1.bar(x, monthly[cname].values, bottom=bottom,
            label=cname, color=CLUSTER_COLORS.get(cname, '#888'),
            alpha=0.85, edgecolor='white', linewidth=0.5)
    bottom += monthly[cname].values

ax1.set_xticks(x)
ax1.set_xticklabels(month_labels, rotation=45, ha='right')
ax1.set_title('월별 불만 트렌드 — 전체 클러스터 (혼재 제외)', fontsize=12, fontweight='bold')
ax1.set_ylabel('불만 문장 수')
ax1.legend(loc='upper left', ncol=2, fontsize=9)

# 2026-03 폭발 표시
if '2026-03' in month_labels:
    idx_peak = month_labels.index('2026-03')
    ax1.axvline(idx_peak, color='red', linestyle='--', linewidth=1.8, alpha=0.7)
    ax1.text(idx_peak + 0.15, ax1.get_ylim()[1] * 0.92,
             '2026-03\n전 클러스터\n동시 폭발', fontsize=9, color='red', fontweight='bold')

# ── 하단: 배송 vs 카메라 꺾은선 대비 ─────────────────────────────
ax2 = axes[1]
delivery_col  = '쿠팡_배송포장'
camera_col    = '제품_카메라+화면'

if delivery_col in monthly.columns and camera_col in monthly.columns:
    ax2.plot(x, monthly[delivery_col].values,
             color=CLUSTER_COLORS[delivery_col], linewidth=2.5,
             marker='o', markersize=5, label=delivery_col)
    ax2.plot(x, monthly[camera_col].values,
             color=CLUSTER_COLORS[camera_col], linewidth=2.5,
             marker='s', markersize=5, label=camera_col)
    ax2.fill_between(x, monthly[delivery_col].values,
                     alpha=0.12, color=CLUSTER_COLORS[delivery_col])
    ax2.fill_between(x, monthly[camera_col].values,
                     alpha=0.12, color=CLUSTER_COLORS[camera_col])

    # 피크 지점 표시
    peak_d_idx = monthly[delivery_col].values.argmax()
    peak_c_idx = monthly[camera_col].values.argmax()
    ax2.annotate(
        f'배송 피크\n{month_labels[peak_d_idx]}',
        xy=(peak_d_idx, monthly[delivery_col].values[peak_d_idx]),
        xytext=(peak_d_idx - 1.5, monthly[delivery_col].values[peak_d_idx] * 1.12),
        fontsize=9, color=CLUSTER_COLORS[delivery_col], fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=CLUSTER_COLORS[delivery_col]),
    )
    ax2.annotate(
        f'카메라 폭발\n{month_labels[peak_c_idx]}\n(4.7× 급증)',
        xy=(peak_c_idx, monthly[camera_col].values[peak_c_idx]),
        xytext=(peak_c_idx - 2.5, monthly[camera_col].values[peak_c_idx] * 1.08),
        fontsize=9, color=CLUSTER_COLORS[camera_col], fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=CLUSTER_COLORS[camera_col]),
    )

ax2.set_xticks(x)
ax2.set_xticklabels(month_labels, rotation=45, ha='right')
ax2.set_title('배송 불만 vs 카메라 불만 — 폭발 시점 비교\n'
              '배송: 출시 직후 피크 → 출시 초기 물류 문제  |  카메라: D+150+ 시한폭탄',
              fontsize=11, fontweight='bold')
ax2.set_ylabel('불만 문장 수')
ax2.legend(fontsize=10)

plt.suptitle('[WHEN] 불만의 타이밍 — 배송과 카메라는 전혀 다른 패턴',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide3_when_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

# 핵심 수치 출력
if camera_col in monthly.columns:
    cam_early   = monthly[camera_col].head(3).mean()
    cam_peak    = monthly[camera_col].max()
    peak_month  = monthly[camera_col].idxmax()
    print(f'카메라 불만: 초기(첫 3개월) 월평균 {cam_early:.0f}건 → {peak_month} {cam_peak:.0f}건 ({cam_peak/cam_early if cam_early>0 else 0:.1f}배 증가)')

---
## [WHO] 소비자 세그먼트 — Slide 4

이메일 도메인 = 디지털 라이프스타일 프록시 (Naver/Gmail/카카오/네이트)

In [ ]:
# ── WHO-1: 이메일 도메인 파싱 ─────────────────────────────────────
def extract_email_domain(member_str):
    if not isinstance(member_str, str):
        return 'unknown'
    # member 컬럼: {'type': 'USER', 'name': '이*연', 'email': 'yyo***@gmail.com'}
    m = re.search(r"'email':\s*'([^']+)'", member_str)
    if not m:
        return 'unknown'
    email = m.group(1)
    return email.split('@')[-1].lower() if '@' in email else 'unknown'

def group_domain(domain):
    if 'naver.com' in domain:                                    return '네이버'
    if 'gmail.com' in domain:                                    return 'Gmail'
    if any(d in domain for d in ('hanmail.net', 'daum.net', 'kakao.com')): return '카카오'
    if 'nate.com' in domain:                                     return '네이트'
    if any(d in domain for d in ('icloud.com', 'me.com')):       return 'iCloud'
    if domain == 'unknown':                                       return '미확인'
    return '기타'

df['email_domain']  = df['member'].apply(extract_email_domain)
df['domain_group']  = df['email_domain'].apply(group_domain)

MAIN_DOMAIN_GROUPS = ['네이버', 'Gmail', '카카오', '네이트']
DOMAIN_COLORS = {
    '네이버': '#03C75A',
    'Gmail':  '#EA4335',
    '카카오': '#FFE812',
    '네이트': '#0055AA',
}

print('도메인 그룹 분포:')
print(df['domain_group'].value_counts().to_string())

In [ ]:
# ── WHO-2: 도메인 그룹 비교 시각화 ───────────────────────────────
df_domain = df[df['domain_group'].isin(MAIN_DOMAIN_GROUPS)].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

dc_list   = MAIN_DOMAIN_GROUPS
bar_color = [DOMAIN_COLORS.get(g, '#888') for g in dc_list]

# 1. 평균 평점
ax1 = axes[0, 0]
domain_rating = df_domain.groupby('domain_group')['rating'].agg(['mean', 'count']).reindex(dc_list)
bars1 = ax1.bar(dc_list, domain_rating['mean'], color=bar_color, edgecolor='white', linewidth=1.5)
ax1.axhline(df['rating'].mean(), color='black', linestyle='--', linewidth=1, alpha=0.5, label='전체 평균')
ax1.set_ylim(3.5, 5.2)
for bar, (m, c) in zip(bars1, domain_rating.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{m:.2f}\n(n={int(c)})', ha='center', va='bottom', fontsize=9)
ax1.set_title('도메인 그룹별 평균 평점', fontsize=12, fontweight='bold')
ax1.set_ylabel('평균 Rating')
ax1.legend(fontsize=9)

# Kruskal-Wallis 유의성
rating_groups = [df_domain[df_domain['domain_group'] == g]['rating'].values for g in dc_list]
H, p_kw = stats.kruskal(*[g for g in rating_groups if len(g) > 0])
ax1.text(0.98, 0.02, f'Kruskal-Wallis p={p_kw:.4f} {"★" if p_kw<0.05 else ""}',
         transform=ax1.transAxes, ha='right', va='bottom', fontsize=8, color='gray')

# 2. 부정 리뷰 비율
ax2 = axes[0, 1]
domain_neg = (df_domain.groupby('domain_group')['is_negative'].mean() * 100).reindex(dc_list)
bars2 = ax2.bar(dc_list, domain_neg.values, color=bar_color, edgecolor='white', linewidth=1.5)
ax2.axhline(df['is_negative'].mean() * 100, color='red', linestyle='--',
            linewidth=1, alpha=0.7, label='전체 평균')
for bar, v in zip(bars2, domain_neg.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
             f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
ax2.set_title('도메인 그룹별 부정 리뷰 비율', fontsize=12, fontweight='bold')
ax2.set_ylabel('부정 비율 (%)')
ax2.legend(fontsize=9)

# 3. 리뷰 길이 (중앙값)
ax3 = axes[1, 0]
domain_len = df_domain.groupby('domain_group')['content_len'].median().reindex(dc_list)
bars3 = ax3.bar(dc_list, domain_len.values, color=bar_color, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars3, domain_len.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{v:.0f}자', ha='center', va='bottom', fontsize=10)
ax3.set_title('도메인 그룹별 리뷰 길이 (중앙값)', fontsize=12, fontweight='bold')
ax3.set_ylabel('리뷰 길이 (자)')

# 4. 도움돼요 (helpfulTrueCount 중앙값)
ax4 = axes[1, 1]
domain_help = df_domain.groupby('domain_group')['helpfulTrueCount'].median().reindex(dc_list)
bars4 = ax4.bar(dc_list, domain_help.values, color=bar_color, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars4, domain_help.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{v:.1f}', ha='center', va='bottom', fontsize=10)
ax4.set_title('도메인 그룹별 도움돼요 (중앙값)\n→ 플랫폼 리뷰 노출 알고리즘 영향력',
              fontsize=11, fontweight='bold')
ax4.set_ylabel('helpfulTrueCount')

plt.suptitle('[WHO] 이메일 도메인 = 디지털 라이프스타일 프록시\n'
             '같은 폰, 다른 시선 — 소비자 유형에 따라 평점·불만·리뷰 패턴이 다르다',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide4_who_domain.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Kruskal-Wallis (평점): H={H:.3f}, p={p_kw:.4f} → {"유의 ★" if p_kw < 0.05 else "비유의"}')

In [ ]:
# ── WHO-3: 도메인 그룹 × 브랜드 선호도 ──────────────────────────
brand_domain = pd.crosstab(
    df_domain['domain_group'],
    df_domain['brand'],
    normalize='index'
) * 100
brand_domain = brand_domain.reindex(MAIN_DOMAIN_GROUPS)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 브랜드 선호 bar
ax1 = axes[0]
x = np.arange(len(MAIN_DOMAIN_GROUPS))
w = 0.35
for i, brand in enumerate(['Apple', 'Samsung']):
    if brand in brand_domain.columns:
        color = '#4A90D9' if brand == 'Apple' else '#E85D5D'
        bars = ax1.bar(x + i * w - w/2, brand_domain[brand].values,
                       w, label=brand, color=color, alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, brand_domain[brand].values):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{v:.0f}%', ha='center', va='bottom', fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels(MAIN_DOMAIN_GROUPS)
ax1.set_title('도메인 그룹별 브랜드 구매 비율', fontsize=12, fontweight='bold')
ax1.set_ylabel('비율 (%)')
ax1.legend()

# Chi-square
chi2_ct = pd.crosstab(df_domain['domain_group'], df_domain['brand'])
chi2, p_chi2, dof, _ = stats.chi2_contingency(chi2_ct.reindex(MAIN_DOMAIN_GROUPS).fillna(0))
ax1.text(0.98, 0.98, f'χ²={chi2:.1f}, p={p_chi2:.4f} {"★" if p_chi2<0.05 else ""}',
         transform=ax1.transAxes, ha='right', va='top', fontsize=8, color='gray')

# 제품별 도메인 분포
ax2 = axes[1]
prod_domain = pd.crosstab(
    df_domain['product_label'],
    df_domain['domain_group'],
    normalize='index'
) * 100
prod_domain = prod_domain.reindex(columns=[c for c in MAIN_DOMAIN_GROUPS if c in prod_domain.columns])
sns.heatmap(prod_domain, annot=True, fmt='.1f', cmap='YlGnBu',
            vmin=0, ax=axes[1], linewidths=0.5,
            cbar_kws={'label': '비율 (%)'})
ax2.set_title('제품 × 도메인 그룹\n(행 기준 %)', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=0)

plt.suptitle('[WHO] 브랜드·제품 선호도 × 도메인 그룹', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide4b_who_brand_domain.png', dpi=150, bbox_inches='tight')
plt.show()

---
## [BRIDGE] WHAT × WHEN × WHO 교차 분석 — Slide 5

이 섹션이 세 개 분석을 **하나의 연구**로 연결하는 핵심입니다.

- **BRIDGE ①**: 코호트(WHEN) × 불만 유형(WHAT) — 얼리어답터와 후기 구매자가 다른 불만을 갖는가?
- **BRIDGE ②**: 도메인 그룹(WHO) × 불만 유형(WHAT) — 소비자 유형별로 어떤 불만이 두드러지는가?

In [ ]:
# ── BRIDGE: 코호트 × 클러스터 + 도메인 × 클러스터 ─────────────────

# 공통 메타 정보 병합 (코호트 + 도메인)
cc_bridge = cc_main.merge(
    df[['reviewId', 'cohort', 'domain_group']],
    on='reviewId', how='left'
)

# ── BRIDGE ①: 코호트 × 불만 클러스터 ─────────────────────────────
cc_b1 = cc_bridge[cc_bridge['cohort'].isin(COHORT_ORDER)]
cohort_cluster_pct = (
    pd.crosstab(cc_b1['cohort'], cc_b1['cluster_name'], normalize='index') * 100
).reindex(index=COHORT_ORDER)
cohort_cluster_pct = cohort_cluster_pct.reindex(
    columns=[c for c in MAIN_CLUSTERS if c in cohort_cluster_pct.columns]
)

# ── BRIDGE ②: 도메인 그룹 × 불만 클러스터 ────────────────────────
cc_b2 = cc_bridge[cc_bridge['domain_group'].isin(MAIN_DOMAIN_GROUPS)]
domain_cluster_pct = (
    pd.crosstab(cc_b2['domain_group'], cc_b2['cluster_name'], normalize='index') * 100
).reindex(index=MAIN_DOMAIN_GROUPS)
domain_cluster_pct = domain_cluster_pct.reindex(
    columns=[c for c in MAIN_CLUSTERS if c in domain_cluster_pct.columns]
)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# BRIDGE ① 히트맵
sns.heatmap(
    cohort_cluster_pct, annot=True, fmt='.1f', cmap='YlOrRd',
    vmin=0, vmax=50, linewidths=0.5, ax=axes[0],
    cbar_kws={'label': '비율 (%)'},
)
axes[0].set_title(
    'BRIDGE ①: 구매 코호트 × 불만 유형 (행 %%)\n'
    '"얼리어답터 → 배송 불만  |  후기 구매자 → 카메라 불만"',
    fontsize=11, fontweight='bold'
)
axes[0].set_ylabel('구매 코호트')
axes[0].tick_params(axis='x', rotation=35)

# BRIDGE ② 히트맵
sns.heatmap(
    domain_cluster_pct, annot=True, fmt='.1f', cmap='YlGnBu',
    vmin=0, vmax=50, linewidths=0.5, ax=axes[1],
    cbar_kws={'label': '비율 (%)'},
)
axes[1].set_title(
    'BRIDGE ②: 소비자 그룹(도메인) × 불만 유형 (행 %%)\n'
    '"동일 제품, 소비자 유형에 따라 불만 구조가 다르다"',
    fontsize=11, fontweight='bold'
)
axes[1].set_ylabel('이메일 도메인 그룹')
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('[BRIDGE] WHAT × WHEN × WHO — 세 분석이 만나는 지점',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'slide5_bridge_crossanalysis.png', dpi=150, bbox_inches='tight')
plt.show()

# 인사이트 출력
print('=== BRIDGE ① 인사이트: 코호트별 최대 불만 ===')
for cohort in COHORT_ORDER:
    if cohort in cohort_cluster_pct.index:
        top = cohort_cluster_pct.loc[cohort].idxmax()
        val = cohort_cluster_pct.loc[cohort].max()
        print(f'  {cohort}: [{top}] {val:.1f}%')

print('\n=== BRIDGE ② 인사이트: 도메인별 최대 불만 ===')
for group in MAIN_DOMAIN_GROUPS:
    if group in domain_cluster_pct.index:
        top = domain_cluster_pct.loc[group].idxmax()
        val = domain_cluster_pct.loc[group].max()
        print(f'  {group}: [{top}] {val:.1f}%')

In [ ]:
# ── BRIDGE 통계 검정 ──────────────────────────────────────────────
from scipy.stats import chi2_contingency

print('=== [통계검정] 코호트 × 클러스터 Chi-square ===')
b1_ct = pd.crosstab(cc_b1['cohort'], cc_b1['cluster_name']).reindex(COHORT_ORDER).fillna(0)
chi2_b1, p_b1, dof_b1, _ = chi2_contingency(b1_ct)
n_b1 = b1_ct.sum().sum()
v_b1 = (chi2_b1 / (n_b1 * (min(b1_ct.shape) - 1))) ** 0.5
print(f'χ²={chi2_b1:.1f}, df={dof_b1}, p={p_b1:.4f}, Cramér\'s V={v_b1:.3f}')
print(f'→ {"코호트별 불만 유형 분포 차이 유의 ★" if p_b1<0.05 else "비유의"}')

print('\n=== [통계검정] 도메인 그룹 × 클러스터 Chi-square ===')
b2_ct = pd.crosstab(cc_b2['domain_group'], cc_b2['cluster_name']).reindex(MAIN_DOMAIN_GROUPS).fillna(0)
chi2_b2, p_b2, dof_b2, _ = chi2_contingency(b2_ct)
n_b2 = b2_ct.sum().sum()
v_b2 = (chi2_b2 / (n_b2 * (min(b2_ct.shape) - 1))) ** 0.5
print(f'χ²={chi2_b2:.1f}, df={dof_b2}, p={p_b2:.4f}, Cramér\'s V={v_b2:.3f}')
print(f'→ {"도메인별 불만 유형 분포 차이 유의 ★" if p_b2<0.05 else "비유의"}')

---
## [SYNTHESIS] 비즈니스 인사이트 종합 — Slide 6

In [ ]:
# ── SYNTHESIS: 핵심 수치 종합 대시보드 ────────────────────────────
coupang_pct  = len(cc[cc['cluster'].isin([1, 2])]) / len(cc) * 100
s26u_cam_pct = (
    cc_main[
        (cc_main['cluster_name'] == '제품_카메라+화면') &
        (cc_main['product_label'] == 'Galaxy S26 Ultra')
    ].shape[0] /
    cc_main[cc_main['product_label'] == 'Galaxy S26 Ultra'].shape[0] * 100
) if cc_main[cc_main['product_label'] == 'Galaxy S26 Ultra'].shape[0] > 0 else 0

cam_early = monthly[camera_col].head(3).mean() if camera_col in monthly.columns else 0
cam_peak  = monthly[camera_col].max()           if camera_col in monthly.columns else 0
multiplier = f'{cam_peak/cam_early:.1f}×' if cam_early > 0 else 'N/A'

# 상위 연쇄 불만
top_chain = cooc.most_common(1)
chain_str = f'{top_chain[0][0][0]} + {top_chain[0][0][1]}: {top_chain[0][1]}건' if top_chain else 'N/A'

fig = plt.figure(figsize=(18, 8))
fig.patch.set_facecolor('#f8f9fa')

# 상단: KPI 카드 4개
kpi_items = [
    ('불만 중 쿠팡 문제',       f'{coupang_pct:.1f}%',  '제품 개선만으로는\n고객 만족 불가'),
    ('S26 Ultra 카메라 불만',   f'{s26u_cam_pct:.1f}%', '마케팅 역설:\n셀링포인트가 최대 불만'),
    ('카메라 불만 증폭',         multiplier,            '출시 초기 모니터링으로\n절대 못 잡는 시한폭탄'),
    ('5점 속 숨겨진 부정 문장',  f'{n_hidden:,}개',       '별점 = 실제 만족도 아님\n문장 단위 분석 필수'),
]
kpi_colors = ['#e74c3c', '#9b59b6', '#f39c12', '#2ecc71']

for i, ((title, value, desc), color) in enumerate(zip(kpi_items, kpi_colors)):
    ax = fig.add_axes([0.04 + i * 0.245, 0.55, 0.22, 0.38])
    ax.set_facecolor(color)
    ax.axis('off')
    ax.text(0.5, 0.75, value, transform=ax.transAxes, ha='center', va='center',
            fontsize=28, fontweight='bold', color='white')
    ax.text(0.5, 0.42, title, transform=ax.transAxes, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(0.5, 0.12, desc, transform=ax.transAxes, ha='center', va='center',
            fontsize=9, color='white', alpha=0.9)

# 하단: 비즈니스 액션 3가지
actions = [
    {
        'axis': 'WHAT\n+ WHEN',
        'title': '신제품 출시 첫 달\n전담 물류·포장 기준 강화',
        'detail': '배송 불만 D+0~30 집중\n쿠팡에 프리미엄 패키징 협의',
        'priority': '🔴 즉시',
        'color': '#fadbd8',
    },
    {
        'axis': 'WHEN',
        'title': '출시 후 6개월 시점\n자동 재모니터링 체계',
        'detail': '카메라 불만은 장기 사용 후 축적\n초기 리뷰만 보면 신호 못 잡음',
        'priority': '🟡 3개월 내',
        'color': '#fef9e7',
    },
    {
        'axis': 'WHO\n+ BRIDGE',
        'title': '소비자 세그먼트별\nCS 대응 우선순위 차별화',
        'detail': '도메인 그룹별 불만 구조 상이\nGmail 사용자 vs 네이버 사용자',
        'priority': '🟢 중장기',
        'color': '#eafaf1',
    },
]

for i, act in enumerate(actions):
    ax = fig.add_axes([0.04 + i * 0.325, 0.04, 0.30, 0.45])
    ax.set_facecolor(act['color'])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.5, 0.90, act['priority'], ha='center', va='top', fontsize=12, fontweight='bold')
    ax.text(0.5, 0.72, act['title'],    ha='center', va='top', fontsize=11, fontweight='bold')
    ax.text(0.5, 0.42, act['detail'],   ha='center', va='top', fontsize=9)
    ax.text(0.5, 0.08, f'분석 축: {act["axis"]}', ha='center', va='bottom',
            fontsize=8, color='gray', style='italic')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_edgecolor('#ccc')

fig.suptitle('[SYNTHESIS] 비즈니스 액션 3가지 — WHAT × WHEN × WHO 통합 인사이트',
             fontsize=14, fontweight='bold', y=0.98)

plt.savefig(OUT_DIR / 'slide6_synthesis.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== 최종 인사이트 요약 ===')
print(f'\n[WHAT] 불만 34%는 쿠팡 문제 (coupang: {coupang_pct:.1f}%)')
print(f'[WHAT] S26 Ultra 카메라 불만 {s26u_cam_pct:.1f}% — 마케팅 역설')
print(f'[WHEN] 카메라 불만 {multiplier} 증폭 — 6개월 후 시한폭탄')
print(f'[FOUNDATION] 5점 리뷰 속 {n_hidden:,}개 숨겨진 불만 — 별점의 한계')
print(f'[BRIDGE] 연쇄 불만 1위: {chain_str}')

In [ ]:
# ── 방법론 요약 (Slide 7 — 발표 마지막) ──────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
ax.axis('off')

pipeline = [
    ('데이터\n2,757 리뷰',   '#ecf0f1', '#2c3e50'),
    ('감성 재라벨링\nTask 2.1 앙상블',  '#d6eaf8', '#154360'),
    ('[WHAT]\n클러스터링\nKMeans k=6\nBERTopic 세분화', '#d5f5e3', '#1a5276'),
    ('[WHEN]\n코호트 분석\n시계열\nBERTopic ToT', '#fef9e7', '#7d6608'),
    ('[WHO]\n도메인 세그먼트\nKruskal-Wallis\nPoisson 회귀', '#f9ebea', '#922b21'),
    ('[BRIDGE]\n교차 분석\nChi-square\nCramér\'s V', '#f5eef8', '#6c3483'),
    ('비즈니스\n인사이트', '#eafaf1', '#1e8449'),
]

n = len(pipeline)
for i, (label, bg, fg) in enumerate(pipeline):
    x0 = i / n
    rect = mpatches.FancyBboxPatch(
        (x0 + 0.005, 0.1), 1/n - 0.015, 0.8,
        boxstyle='round,pad=0.02',
        facecolor=bg, edgecolor='#bbb', linewidth=1.5,
        transform=ax.transAxes, clip_on=False
    )
    ax.add_patch(rect)
    ax.text(
        x0 + 0.5/n, 0.5, label,
        transform=ax.transAxes, ha='center', va='center',
        fontsize=8.5, color=fg, fontweight='bold',
        multialignment='center'
    )
    if i < n - 1:
        ax.annotate('', xy=((i+1)/n + 0.002, 0.5), xytext=((i+1)/n - 0.002, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#888', lw=1.5))

ax.set_title('분석 파이프라인 — Task 2.1 기반 감성 모델 → 6가지 분석 축',
             fontsize=13, fontweight='bold', pad=20)
plt.savefig(OUT_DIR / 'slide7_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== 생성된 발표용 이미지 파일 목록 ===')
for f in sorted(OUT_DIR.glob('*.png')):
    print(f'  {f.name}')